# OCR Agent

This notebook is about how to build OCR Agents using langchain and Gemini.

Gemini is very great to read files, I have good results using it.

There are two options when we have to work with images:
- Read image as files;
- Read image as bytes.

# 1 - Reading file

## Option 1: File as image

### Step 1: Read files

In [ ]:
def read_file_as_bytes(file_path):
    try:
        with open(file_path, 'rb') as file:
            file_bytes = file.read()
            return file_bytes
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return None
    except Exception as e:
        print(f"Error reading file: {e}")
        return None

# Usage examples
# pdf_bytes = read_file_as_bytes('document.pdf')
# jpg_bytes = read_file_as_bytes('photo.jpg')
# png_bytes = read_file_as_bytes('image.png')

### Step 2: Preprocessing images (if needed)

In [ ]:
from PIL import Image, ImageEnhance
import numpy as np
import cv2

def preprocess_image(pil_image: Image.Image) -> Image.Image:
    """
    Apply preprocessing techniques to a better quality of the image for OCR
    """
    # 1. Convert image to grayscale
    img = pil_image.convert("L")

    # 2. Increase contrast and bright (Pillow)
    enhancer_contrast = ImageEnhance.Contrast(img)
    img = enhancer_contrast.enhance(1.5) # increase contrast in 50%
    enhancer_brightness = ImageEnhance.Brightness(img)
    img = enhancer_brightness.enhance(1.1) # increase bright in 10%

    # 3. Binarization
    # Converting to NumPy array before use OpenCV
    np_img = np.array(img)
  
    # Optional: Noise Removal (OpenCV) - Use Denoising or Bilateral Filter
    # If the image has a lot of noise (pixelated), this can help.
    denoised_img = cv2.fastNlMeansDenoising(np_img, None, 10, 7, 21) # To images in grayscale
    binary_img = denoised_img

    # Returns to PIL image
    processed_pil_image = Image.fromarray(binary_img)

    return processed_pil_image

### Step 3: Instantiate the Gemini LLM, preprocess the file and invoke LLM

In [ ]:
import base64
from io import BytesIO
import fitz  # PyMuPDF
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI

# Instantiating LLM, Google gives you a few tokens per day to use it for free
llm = ChatGoogleGenerativeAI(api_key='here-goes-your-api-key',
            model="gemini-2.5-flash", # make sure this model still exists, because it changes a lot
            temperature=0,
            max_tokens=10000,
            timeout=None,
            max_retries=2,
            # other params...
        )

# Read file
resp_file = read_file_as_bytes('document.pdf') # Supposing that your file is a .pdf

pdf_bytes = BytesIO(resp_file)

# Here we'll pass through each page in your file and read it
doc = fitz.open(stream=pdf_bytes)
processed_images_base64 = [] # A list to save the images
for page_num in range(doc.page_count): # This loop pass through all the pages of the file
    page = doc.load_page(page_num) # Loading the page
    pix = page.get_pixmap(matrix=fitz.Matrix(2, 2)) # Renders at a higher DPI (2x) for better quality
    pil_image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples) # Converts bytes to RGB image
    
    processed_pil_image = preprocess_image(pil_image) # Preprocessing the image
    
    img_byte_arr = BytesIO()
    processed_pil_image.save(img_byte_arr, format="JPEG") # Temporary file to save the image
    processed_images_base64.append(base64.b64encode(img_byte_arr.getvalue()).decode("utf-8"))
doc.close()

# Now we must prepare the text and images as input to invoke the LLM
prompt = "summarizes all the relevant information in this file"
messages_to_send = [{"type": "text", "text": prompt}]

for img_b64 in processed_images_base64: # Passing through all images from the file and save
    messages_to_send.append({
        "type": "image_url",
        "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}
    })

# Prepare the message
message = HumanMessage(content=messages_to_send)

response = llm.invoke([message])

extracted_content = response.content # This is the response
print(extracted_content)

# Option 2: File as bytes

### Step 1: Instantiate Gemini LLM and set image for OCR

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key="your_Google_API_KEY")

# Taking advantage of the function we built above 'read_file_as_bytes'
resp_file = read_file_as_bytes('document.pdf') # Supposing that your file is a .pdf

# Detect file type
if resp_file.startswith(b'%PDF'):
    mime_type = 'application/pdf'
elif resp_file.startswith(b'\xff\xd8'):
    mime_type = 'image/jpeg'
elif resp_file.startswith(b'\x89PNG'):
    mime_type = 'image/png'
else:
    mime_type = 'application/octet-stream'

# We created a 'Part' (part of the content) directly from the bytes
document = types.Part.from_bytes(
    data=resp_file,
    mime_type=mime_type
)

# The content sent will be your prompt + file (bytes)
contents_to_send = [
    document,
    """Summarizes all the relevant information in this file. 
    If there is more than 1 page, read one at a time to avoid confusing the information"""
]

# Here you configure the hyperparameters and instructions for LLM 
config_doc = types.GenerateContentConfig(
    temperature=0.0,
    max_output_tokens=10000,
    system_instruction='You are an excelent content extractor' # Here you set what's the purpose of LLM
)

# Calling the LLM and get response
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=contents_to_send,
    config=config_doc
)

extracted_content = response.text

Notice that we don’t need to convert the file (bytes) into a image. Therefore, we preserve all the original information from the file, so the LLM can better “understand” the content.

# 2 - Building Agent

In [ ]:
from google import genai
from google.genai import types
from langchain_core.tools import tool

client = genai.Client(api_key="your_Google_API_KEY")

@tool
def ocr(file_path: str) -> str:
    """
    This tool is for OCR (Optical Character Recognition).
    Use this tool to read .pdf, .png and .jpg files.
    
    input: file path (str)
    output: response (str)
    """
    
    # Read the file as bytes
    with open(file_path, 'rb') as f:
        resp_file = f.read()

    # Detect file type
    if resp_file.startswith(b'%PDF'):
        mime_type = 'application/pdf'
    elif resp_file.startswith(b'\xff\xd8'):
        mime_type = 'image/jpeg'
    elif resp_file.startswith(b'\x89PNG'):
        mime_type = 'image/png'
    else:
        mime_type = 'application/octet-stream'
    
    # We created a 'Part' (part of the content) directly from the bytes
    document = types.Part.from_bytes(
        data=resp_file,
        mime_type=mime_type
    )
    
    # The content sent will be your prompt + file (bytes)
    contents_to_send = [
        document,
        """Summarizes all the relevant information in this file. 
        If there is more than 1 page, read one at a time to avoid confusing the information"""
    ]
    
    # Here you configure the hyperparameters and instructions for LLM 
    config_doc = types.GenerateContentConfig(
        temperature=0.0,
        max_output_tokens=10000,
        system_instruction='You are an excelent content extractor' # Here you set what's the purpose of LLM
    )
    
    # Calling the LLM and get response
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=contents_to_send,
        config=config_doc
    )
    
    extracted_content = response.text
    return extractend_content

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm_genai = ChatGoogleGenerativeAI(api_key="your_Google_API_KEY",
            model="gemini-2.5-flash",
            temperature=0,
            max_tokens=None,
            timeout=None,
            max_retries=2,
            # other params...
        )

agent = create_react_agent(
    llm_genai,
    tools=[ocr]
)

result = agent.invoke({
    "messages": [("user", "Extract text from this file: /path/to/your/file.pdf")]
})

print(result)